In [1]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [2]:
def run_sql_file(path):                         # one .sql file
    with open(path, 'r') as f:                  # read only
        script = f.read()                       # whole file
    con.executescript(script)                   # every statement
    con.commit()                                # save to the .db
    print('ran', path)  

In [4]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\01_revenue_customer.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\01_revenue_customer.sql


In [5]:
q("""SELECT
    COUNT(*) AS total_customers,
    ROUND(AVG(arpu_inr), 2) AS average_recorded_arpu,
    COUNT(DISTINCT arpu_inr) AS distinct_arpu_values
FROM customers;""")

,total_customers,average_recorded_arpu,distinct_arpu_values
0,22000,421.65,13


In [6]:
q("""SELECT
    COUNT(*) AS customers_without_payments
FROM customers AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM recharges AS r
    WHERE r.customer_id = c.customer_id
      AND r.recharge_date >= '2025-01-01'
      AND r.recharge_date < '2025-07-01'
);""")

,customers_without_payments
0,3711


In [8]:
q("""SELECT name, type
FROM sqlite_master
WHERE name = 'v_revenue_customer';""")

,name,type
0,v_revenue_customer,view


In [ ]:
q("""SELECT *
FROM v_revenue_customer;""")

,customer_id,recharge_count,recharge_total_6m,actual_arpu_6m
0,1,3,897,149.50
1,2,2,198,33.00
2,3,6,5073,845.50
3,4,0,0,0.00
4,5,2,1438,239.67
...,...,...,...,...
21995,21996,6,10478,1746.33
21996,21997,1,719,119.83
21997,21998,5,1045,174.17
21998,21999,2,798,133.00


In [11]:
q("""SELECT
    SUM(CASE WHEN recharge_count > 0 THEN 1 ELSE 0 END)
        AS customers_with_payments,
    SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END)
        AS customers_without_payments
FROM v_revenue_customer;""")

,customers_with_payments,customers_without_payments
0,18289,3711


In [13]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\02_usage_customer.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\02_usage_customer.sql


In [14]:
q("""SELECT * FROM v_usage_customer;""")

,customer_id,usage_days,total_data_mb,total_voice_min,total_sms,roaming_days,mb_per_active_day,mb_per_observed_day,last_usage_date,recency_days
0,1,17,12405.0,333,3,0,729.71,68.54,2025-04-26,65.0
1,2,11,2822.5,424,6,1,256.59,15.59,2025-02-11,139.0
2,3,55,102791.3,2205,23,1,1868.93,567.91,2025-06-25,5.0
3,4,40,8755.2,2262,26,0,218.88,48.37,2025-03-16,106.0
4,5,39,44725.0,1364,9,0,1146.79,247.10,2025-06-18,12.0
...,...,...,...,...,...,...,...,...,...,...
21995,21996,79,522264.4,4713,51,2,6610.94,2885.44,2025-06-30,0.0
21996,21997,101,141403.8,6621,83,2,1400.04,781.24,2025-06-28,2.0
21997,21998,65,4881.9,2111,30,6,75.11,26.97,2025-06-30,0.0
21998,21999,18,20771.6,596,11,0,1153.98,114.76,2025-03-31,91.0


In [15]:
q("""SELECT * FROM usage_daily;""")

,usage_id,customer_id,usage_date,data_mb,voice_minutes,sms_count,is_roaming
0,1,1,2025-01-04,571.8,29,0,0
1,2,1,2025-01-13,650.7,31,0,0
2,3,1,2025-02-05,704.6,42,0,0
3,4,1,2025-02-06,1012.0,6,0,0
4,5,1,2025-02-15,2150.4,58,0,0
...,...,...,...,...,...,...,...
1326734,1326735,22000,2025-06-24,1321.7,15,1,0
1326735,1326736,22000,2025-06-26,1151.5,63,0,0
1326736,1326737,22000,2025-06-27,2867.2,107,1,1
1326737,1326738,22000,2025-06-28,1231.6,186,3,0


In [23]:
q("""SELECT COUNT(*) AS subs                         -- who is left?
FROM   complaints c                           -- all 22,000
 LEFT JOIN  (SELECT customer_id, COUNT(*) AS n        -- tickets each
       FROM customers GROUP BY customer_id) x -- rolled up first
       ON x.customer_id = c.customer_id;   """)

,subs
0,26841


In [17]:
q("""SELECT
    COUNT(DISTINCT customer_id) AS customers_with_complaints
FROM complaints;""")

,customers_with_complaints
0,15369


In [26]:
q("""SELECT plan_id, data_quota, daily_data_gb      -- text and number
FROM   plans                                 -- 14 plans
WHERE  plan_id IN (5, 11); """)

,plan_id,data_quota,daily_data_gb
0,5,2.5 GB/day,2.5
1,11,75 GB/month,2.5


In [25]:
q("""SELECT
    COUNT(*) AS customers_without_complaints
FROM customers AS c
LEFT JOIN complaints AS cp
    ON c.customer_id = cp.customer_id
WHERE cp.customer_id IS NULL;""")

,customers_without_complaints
0,6631


In [28]:
q("""SELECT
    COUNT(*) AS customers_without_complaints
FROM customers AS c
LEFT JOIN complaints AS cp
    ON c.customer_id = cp.customer_id
WHERE cp.customer_id IS NULL;""")

,customers_without_complaints
0,6631


In [ ]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\03_complaints_customer.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\03_complaints_customer.sql


In [31]:
q("""SELECT COUNT(*) FROM v_complaint_customer;""")

,COUNT(*)
0,22000


In [30]:
q("""SELECT * FROM v_complaint_customer;""")

,customer_id,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,1,1,1,0,NaN
1,2,1,1,0,NaN
2,3,0,0,0,NaN
3,4,0,0,0,NaN
4,5,2,1,1,13.0
...,...,...,...,...,...
21995,21996,2,1,1,11.0
21996,21997,1,0,1,13.0
21997,21998,1,0,1,10.0
21998,21999,0,0,0,NaN


In [33]:
q("""SELECT COUNT(*) 
  FROM v_complaint_customer
  WHERE complaint_count=0;""")

,COUNT(*)
0,6631


In [34]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\04_plans_clean.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\04_plans_clean.sql


In [35]:
q("""SELECT * FROM v_plans_clean;""")

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,quota_basis,quota_gb,monthly_gb,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,per day,0.5,15.0,None,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,per day,1.0,30.0,None,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,per day,1.0,30.0,None,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,per day,1.5,45.0,None,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,per day,2.5,75.0,None,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,per day,2.5,75.0,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,per day,1.5,45.0,None,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,per day,2.0,60.0,None,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,per day,2.0,60.0,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,per month,40.0,40.0,OTT-Lite,1


In [36]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\05_usage_month.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\05_usage_month.sql


In [37]:
q("""SELECT * FROM v_usage_month;""")

,customer_id,month,usage_days,total_mb,total_voice,roaming_days
0,1,2025-01,2,1222.5,60,0
1,1,2025-02,4,4496.3,114,0
2,1,2025-03,5,2871.0,50,0
3,1,2025-04,6,3815.2,109,0
4,2,2025-01,7,1870.5,225,1
...,...,...,...,...,...,...
117827,22000,2025-02,20,29489.8,1435,2
117828,22000,2025-03,20,33024.3,1489,0
117829,22000,2025-04,17,23536.9,1170,1
117830,22000,2025-05,23,39269.1,1521,1


In [38]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\06_customer_features.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\06_customer_features.sql


In [ ]:
q(""")